In [1]:
import pandas as pd
import requests
from io import StringIO
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import re
import numpy as np
import plotly.express as px

## ***Coletando os dados de cada jogo***

### Função para coletar os dados do League of Leagends

In [3]:
headers = {
    'User-Agent': 'Mozilla/5.0'
}

def coletar_lol(url, nome_csv):

    response = requests.get(url, headers=headers)

    soup = BeautifulSoup(response.text, 'html.parser')

    texto = soup.get_text(' ', strip=True)

    padrao = (
        r'(20\d{2}|2011|2012|2013|2014|2015|2016|2017|2018|2019)'
        r'\s+(\d+)'
        r'\s+Million'
        r'\s*([+-]?\d+(?:\.\d+)?%)'
    )

    dados = re.findall(padrao, texto)

    df = pd.DataFrame(
        dados,
        columns=[
            'Ano',
            'Jogadores_Ativos_Mensais',
            'Crescimento_Percentual'
        ]
    )

    df['Ano'] = df['Ano'].astype(int)

    df['Jogadores_Ativos_Mensais'] = (
        df['Jogadores_Ativos_Mensais']
        .astype(float)
    )

    df['Crescimento_Percentual'] = (
        df['Crescimento_Percentual']
        .str.replace('%', '', regex=False)
        .astype(float)
    )

    df.to_csv(
        nome_csv,
        index=False,
        encoding='utf-8-sig'
    )

    return df

### Função para coletar os dados do site Steamplayercount

In [4]:
def coletar_steam(url, nome_csv):

    response = requests.get(url, headers=headers)

    html = StringIO(response.text)

    tabelas = pd.read_html(html)

    df = tabelas[1].copy()

    df = df.rename(columns={
        'Month': 'Mês',
        'Peak': 'Pico',
        'Gain': 'Ganho',
        '% Gain': 'Ganho%',
        'Min Daily Peak': 'Pico_Mínimo_Diário',
        'Avg Daily Peak': 'Pico_Médio_Diário'
    })

    df.to_csv(
        nome_csv,
        index=False,
        encoding='utf-8-sig'
    )

    return df

### Coleta dos dados

In [6]:
print('Coletando League of Legends...')

df_lol = coletar_lol(
    'https://turbosmurfs.gg/article/league-of-legends-player-count-and-statistics',
    'jogadores_lol.csv'
)

print('League of Legends concluído!')


print('Coletando Dota 2...')

df_dota2 = coletar_steam(
    'https://steamplayercount.com/app/570',
    'jogadores_dota2.csv'
)

print('Dota 2 concluído!')


print('Coletando SMITE...')

df_smite = coletar_steam(
    'https://steamplayercount.com/app/386360',
    'jogadores_smite.csv'
)

print('SMITE concluído!')

print('\nTodos os arquivos CSV foram criados com sucesso!')

Coletando League of Legends...
League of Legends concluído!
Coletando Dota 2...
Dota 2 concluído!
Coletando SMITE...
SMITE concluído!

Todos os arquivos CSV foram criados com sucesso!


## ***Leitura e Tratamento dos arquivos CSV***

In [7]:
df_lol = pd.read_csv('jogadores_lol.csv')
df_dota2 = pd.read_csv('jogadores_dota2.csv')
df_smite = pd.read_csv('jogadores_smite.csv')

### League of Legends

In [8]:
df_lol['Ano'] = df_lol['Ano'].astype(int)

df_lol['Jogadores_Ativos_Mensais'] = (
    df_lol['Jogadores_Ativos_Mensais']
    .astype(float)
)

df_lol['Crescimento_Percentual'] = (
    df_lol['Crescimento_Percentual']
    .astype(float)
)

# Salva as alterações
df_lol.to_csv(
    'jogadores_lol.csv',
    index=False,
    encoding='utf-8-sig'
)

In [11]:
df_lol.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Ano                       15 non-null     int64  
 1   Jogadores_Ativos_Mensais  15 non-null     float64
 2   Crescimento_Percentual    15 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 492.0 bytes


In [14]:
df_lol.head(25)

,Ano,Jogadores_Ativos_Mensais,Crescimento_Percentual
0,2011,15.0,200.0
1,2012,32.0,113.0
2,2013,67.0,110.0
3,2014,70.0,4.5
4,2015,90.0,28.0
5,2016,100.0,11.0
6,2017,81.0,-18.9
7,2018,75.0,-7.5
8,2019,116.0,54.5
9,2020,137.0,18.0


### Dota 2

In [10]:
df_dota2['Mês'] = pd.to_datetime(df_dota2['Mês'])

df_dota2['Ano'] = df_dota2['Mês'].dt.year

df_dota2_anual = (
    df_dota2
    .groupby('Ano', as_index=False)['Pico']
    .mean()
)

df_dota2_anual['Crescimento_Percentual'] = (
    df_dota2_anual['Pico']
    .pct_change()
    .mul(100)
    .round(0)
    .fillna(0)
)

df_dota2_anual.to_csv(
    'jogadores_dota2_anual.csv',
    index=False,
    encoding='utf-8-sig'
)

C:\Users\drrec\AppData\Local\Temp\ipykernel_13168\1038561284.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_dota2['Mês'] = pd.to_datetime(df_dota2['Mês'])


In [15]:
df_dota2.info()

<class 'pandas.DataFrame'>
RangeIndex: 174 entries, 0 to 173
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Mês                 174 non-null    datetime64[us]
 1   Pico                174 non-null    int64         
 2   Ganho               174 non-null    str           
 3   Ganho%              174 non-null    str           
 4   Pico_Mínimo_Diário  174 non-null    int64         
 5   Pico_Médio_Diário   174 non-null    int64         
 6   Ano                 174 non-null    int32         
dtypes: datetime64[us](1), int32(1), int64(3), str(2)
memory usage: 9.0 KB


In [16]:
df_dota2.head()

,Mês,Pico,Ganho,Ganho%,Pico_Mínimo_Diário,Pico_Médio_Diário,Ano
0,2026-02-01,870671,+13161,+2%,743693,788656,2026
1,2026-01-01,857510,-47739,-5%,705157,771469,2026
2,2025-12-01,905249,-3840,-1%,687249,800206,2025
3,2025-11-01,909089,+45253,+5%,699238,798545,2025
4,2025-10-01,863836,-83252,-9%,716015,767147,2025


### Smite

In [17]:
df_smite['Mês'] = pd.to_datetime(df_smite['Mês'])

df_smite['Ano'] = df_smite['Mês'].dt.year

df_smite_anual = (
    df_smite
    .groupby('Ano', as_index=False)['Pico']
    .mean()
)

df_smite_anual['Crescimento_Percentual'] = (
    df_smite_anual['Pico']
    .pct_change()
    .mul(100)
    .round(0)
    .fillna(0)
)

df_smite_anual.to_csv(
    'jogadores_smite_anual.csv',
    index=False,
    encoding='utf-8-sig'
)

C:\Users\drrec\AppData\Local\Temp\ipykernel_13168\1586720351.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_smite['Mês'] = pd.to_datetime(df_smite['Mês'])


In [18]:
df_smite.info()

<class 'pandas.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Mês                 126 non-null    datetime64[us]
 1   Pico                126 non-null    int64         
 2   Ganho               126 non-null    str           
 3   Ganho%              126 non-null    str           
 4   Pico_Mínimo_Diário  126 non-null    int64         
 5   Pico_Médio_Diário   126 non-null    int64         
 6   Ano                 126 non-null    int32         
dtypes: datetime64[us](1), int32(1), int64(3), str(2)
memory usage: 6.5 KB


In [19]:
df_smite.head()

,Mês,Pico,Ganho,Ganho%,Pico_Mínimo_Diário,Pico_Médio_Diário,Ano
0,2026-02-01,2238,-351,-14%,1703,1884,2026
1,2026-01-01,2589,+287,+13%,1912,2148,2026
2,2025-12-01,2302,-4,0%,1703,1943,2025
3,2025-11-01,2306,+44,+2%,1763,1948,2025
4,2025-10-01,2262,-36,-2%,1655,1886,2025


## ***Padronização dos DataFrames***

In [ ]:
#League of Legends

df_lol_anual = df_lol.rename(
    columns={
        'Jogadores_Ativos_Mensais': 'Valor'
    }
)

df_lol_anual['Jogo'] = 'League of Legends'

df_lol_anual = df_lol_anual[
    ['Ano', 'Valor', 'Jogo']
]

#Dota 2

df_dota2_anual = df_dota2_anual.rename(
    columns={
        'Pico': 'Valor'
    }
)

df_dota2_anual['Jogo'] = 'Dota 2'

df_dota2_anual = df_dota2_anual[
    ['Ano', 'Valor', 'Jogo']
]

#SMITE

df_smite_anual = df_smite_anual.rename(
    columns={
        'Pico': 'Valor'
    }
)

df_smite_anual['Jogo'] = 'SMITE'

df_smite_anual = df_smite_anual[
    ['Ano', 'Valor', 'Jogo']
]

### Unindo so DataFrames

In [23]:
df_todos = pd.concat(
    [
        df_lol_anual,
        df_dota2_anual,
        df_smite_anual
    ],
    ignore_index=True
)

df_todos = df_todos.sort_values(
    ['Jogo', 'Ano']
)

df_todos.to_csv(
    'comparacao_mobas.csv',
    index=False,
    encoding='utf-8-sig'
)

### Conversão para base 100

In [24]:
ano_base = 2015
ano_limite = 2025

df_todos = df_todos[
    df_todos['Ano'].between(
        ano_base,
        ano_limite
    )
]

df_todos = df_todos.sort_values(
    ['Jogo', 'Ano']
)

df_todos['Base_100'] = (
    df_todos
    .groupby('Jogo')['Valor']
    .transform(
        lambda x: x / x.iloc[0] * 100
    )
)

df_todos.to_csv(
    'comparacao_base100.csv',
    index=False,
    encoding='utf-8-sig'
)

In [25]:
df_todos.info()

<class 'pandas.DataFrame'>
Index: 33 entries, 19 to 41
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Ano       33 non-null     int64  
 1   Valor     33 non-null     float64
 2   Jogo      33 non-null     str    
 3   Base_100  33 non-null     float64
dtypes: float64(2), int64(1), str(1)
memory usage: 1.3 KB


In [26]:
df_todos.head(33)

,Ano,Valor,Jogo,Base_100
19,2015,9.852034e+05,Dota 2,100.000000
20,2016,1.114169e+06,Dota 2,113.090241
21,2017,9.121416e+05,Dota 2,92.584086
22,2018,7.856926e+05,Dota 2,79.749275
23,2019,8.595110e+05,Dota 2,87.241983
24,2020,7.137580e+05,Dota 2,72.447780
25,2021,7.038941e+05,Dota 2,71.446574
26,2022,7.930892e+05,Dota 2,80.500042
27,2023,7.573639e+05,Dota 2,76.873862
28,2024,7.876488e+05,Dota 2,79.947830


## ***Uma breve analize***

In [27]:
df_todos = pd.read_csv('comparacao_base100.csv')

fig = px.line(
    df_todos,
    x='Ano',
    y='Base_100',
    color='Jogo',
    markers=True,
    color_discrete_map={
        'League of Legends': '#DF1717',
        'Dota 2': '#22C265',
        'SMITE': '#46C0E5'
    }
)

fig.update_layout(
    title='Comparação da evolução dos MOBAs (Índice Base 100)',
    xaxis_title='Ano',
    yaxis_title='Índice Base 100',
    template='plotly_white',
    hovermode='x unified'
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=8)
)

fig.show()

O gráfico evidencia que, entre 2015 e 2018, os três jogos apresentaram uma tendência de redução em relação ao ano-base. A partir de 2019, observa-se uma recuperação mais acentuada do League of Legends, que alcança o maior crescimento relativo entre os jogos analisados e mantém esse desempenho até o final da série histórica, apesar de uma leve redução após 2023. O SMITE também apresentou crescimento entre 2019 e 2021, porém sofreu uma queda contínua nos anos seguintes, encerrando o período com desempenho inferior ao do ano-base. Já o Dota 2 apresentou oscilações menores e manteve um comportamento relativamente estável ao longo do período. Dessa forma, os resultados não fornecem evidências de que o League of Legends esteja em processo de decadência quando comparado aos demais MOBAs analisados. Pelo contrário, entre os jogos considerados, foi o que apresentou a evolução relativa mais favorável durante o período analisado.

Lembrando que a Riot Games não disponibiliza os dados de seus jogos, e o que foi usado é uma estimativa retirado do site turbosmurfs.gg